In [183]:
import os
import json
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

In [184]:
with open("data/raw/statements.json","r",encoding="utf-8") as f:
    data=json.load(f)

In [185]:
data

[{'url': 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20210127a.htm',
  'date': '2021-01-27',
  'doc_type': 'statement',
  'text': "The Federal Reserve is committed to using its full range of tools to support the U.S. economy in this challenging time, thereby promoting its maximum employment and price stability goals.\n\nThe COVID-19 pandemic is causing tremendous human and economic hardship across the United States and around the world. The pace of the recovery in economic activity and employment has moderated in recent months, with weakness concentrated in the sectors most adversely affected by the pandemic. Weaker demand and earlier declines in oil prices have been holding down consumer price inflation. Overall financial conditions remain accommodative, in part reflecting policy measures to support the economy and the flow of credit to U.S. households and businesses.\n\nThe path of the economy will depend significantly on the course of the virus, including progre

In [186]:
word_count =[len(statement["text"].split()) for statement in data]
print(min(word_count), max(word_count), sum(word_count) / len(word_count))

131 614 351.6458333333333


In [187]:
paragram_count = [ len(text['text'].split("\n\n")) for text in data]
print(min(paragram_count), max(paragram_count), sum(paragram_count) / len(paragram_count))

3 6 5.104166666666667


In [188]:
para_words = [len(p.split()) for s in data for p in s["text"].split("\n\n")]
print(min(para_words), max(para_words), sum(para_words) / len(para_words))

17 314 68.89387755102041


In [189]:
for s in data:
    for i,p in enumerate(s['text'].split("\n\n")):
        if len(p.split()) >180:
            print(s['date'],i, p[:80])

2021-01-27 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2
2021-03-17 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2
2021-04-28 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2
2021-06-16 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2
2021-07-28 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2
2021-09-22 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2
2021-11-03 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2
2021-12-15 3 The Committee seeks to achieve maximum employment and inflation at the rate of 2


In [190]:
def split_long_paragraph(text,max_words=30):
    chunk=[]
    sentence_to_chunk=''
    
    sent_token=sent_tokenize(text)
    for sent in sent_token:
        current=len(sent.split())      
       
        if len(sentence_to_chunk.split())+current>max_words and sentence_to_chunk != '':            
            chunk.append(sentence_to_chunk.strip())
            sentence_to_chunk=''
        sentence_to_chunk+=" "+sent
            
    if sentence_to_chunk != '':
        chunk.append(sentence_to_chunk.strip())
        
    return chunk
        

In [191]:
total_chunk=[]
for d in data:
    chunk_counter+=1
    for i,p in enumerate(d['text'].split('\n\n')):
        pieces = split_long_paragraph(p, max_words=150)
        for j,p in enumerate(pieces):
            chunk_id=str(d['date'])+"_p"+str(i+1)+"_c"+str(j+1)
            chunk_dict={"chunk_id":chunk_id,"date":d['date'],"doc_type":d["doc_type"],"url":d["url"],"para_index":i+1,
                    "chunk_index":j+1,"text":p }
            total_chunk.append(chunk_dict)
               
    


In [192]:
total_chunk

[{'chunk_id': '2021-01-27_p1_c1',
  'date': '2021-01-27',
  'doc_type': 'statement',
  'url': 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20210127a.htm',
  'para_index': 1,
  'chunk_index': 1,
  'text': 'The Federal Reserve is committed to using its full range of tools to support the U.S. economy in this challenging time, thereby promoting its maximum employment and price stability goals.'},
 {'chunk_id': '2021-01-27_p2_c1',
  'date': '2021-01-27',
  'doc_type': 'statement',
  'url': 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20210127a.htm',
  'para_index': 2,
  'chunk_index': 1,
  'text': 'The COVID-19 pandemic is causing tremendous human and economic hardship across the United States and around the world. The pace of the recovery in economic activity and employment has moderated in recent months, with weakness concentrated in the sectors most adversely affected by the pandemic. Weaker demand and earlier declines in oil prices have been holdi

In [193]:
print(len(total_chunk))
print(max([len(c['text'].split()) for c in total_chunk]))
print(len(set(c["chunk_id"] for c in total_chunk)) == len(total_chunk))

260
150
True


In [194]:
os.makedirs("data/processed",exist_ok=True)
with open("data/processed/chunk.json","w",encoding="utf-8") as f:
    json.dump(total_chunk,f,ensure_ascii=False, indent=2)